# 03b: RegEx with Pandas for Data Cleaning

This notebook covers how to use Regular Expressions with Pandas for real-world data cleaning tasks.

**Prerequisites:**
- Completed 03a_RegEx_Fundamentals.ipynb
- Basic Pandas knowledge (Series, DataFrames)

**Learning Objectives:**
- Use `.str` accessor methods with regex
- Filter DataFrames using pattern matching
- Extract structured data from messy text
- Clean and transform data using regex substitution

**Duration:** ~60 minutes

---

In [ ]:
import pandas as pd
import re
import numpy as np

print("Pandas version:", pd.__version__)

## 1. Pandas String Methods Overview

Pandas provides the `.str` accessor for vectorized string operations on Series.

### Key Regex Methods:

| Method | Description | Returns |
|--------|-------------|----------|
| `.str.contains(pattern)` | Check if pattern exists | Boolean Series |
| `.str.match(pattern)` | Check if pattern matches from start | Boolean Series |
| `.str.extract(pattern)` | Extract first match groups | DataFrame |
| `.str.extractall(pattern)` | Extract all matches | MultiIndex DataFrame |
| `.str.findall(pattern)` | Find all matches | Series of lists |
| `.str.replace(pattern, repl)` | Replace pattern | Series |
| `.str.split(pattern)` | Split by pattern | Series of lists |

## 2. Filtering with `.str.contains()`

The most common regex operation: filtering rows based on pattern matching.

In [ ]:
# Sample dataset: Customer feedback
feedback_data = {
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'feedback': [
        'Great product! Love it.',
        'Product arrived damaged. Very disappointed.',
        'Fast delivery, good quality.',
        'Not worth the price. Poor quality.',
        'Excellent customer service!',
        'The product broke after 2 days.',
        'Best purchase ever! Highly recommend.',
        'Poor packaging, item was broken.'
    ]
}

df = pd.DataFrame(feedback_data)
df

In [ ]:
# Find feedback containing negative words
negative_pattern = r'damaged|disappointed|poor|broke|broken'

negative_feedback = df[df['feedback'].str.contains(negative_pattern, case=False, regex=True)]
print("Negative feedback:")
negative_feedback

In [ ]:
# Find feedback with exclamation marks (enthusiastic customers)
enthusiastic = df[df['feedback'].str.contains(r'!', regex=True)]
print("Enthusiastic feedback:")
enthusiastic

### Handling NaN Values

Use `na=False` to handle missing values gracefully.

In [ ]:
# Data with missing values
df_with_nan = df.copy()
df_with_nan.loc[2, 'feedback'] = np.nan

# Without na=False, this would raise an error or return NaN
# With na=False, NaN rows are treated as False
positive = df_with_nan[df_with_nan['feedback'].str.contains(r'great|excellent|best', 
                                                            case=False, 
                                                            na=False,
                                                            regex=True)]
positive

## 3. Extracting Data with `.str.extract()`

Extract structured information from text using capturing groups `()`.

In [ ]:
# Sample: Contact information
contacts = pd.DataFrame({
    'info': [
        'John Smith - john.smith@email.com - 012-3456789',
        'Jane Doe - jane_doe@company.org - 011-9876543',
        'Bob Wilson - bob123@gmail.com - 019-1112233',
        'Alice Lee - alice.lee@uni.edu.my - 013-4445556'
    ]
})

contacts

In [ ]:
# Extract email addresses
# Pattern: word characters, dots, underscores @ domain
email_pattern = r'([\w\.]+@[\w\.]+)'

contacts['email'] = contacts['info'].str.extract(email_pattern)
contacts

In [ ]:
# Extract phone numbers (Malaysian format: 01X-XXXXXXX)
phone_pattern = r'(01\d-\d{7})'

contacts['phone'] = contacts['info'].str.extract(phone_pattern)
contacts

In [ ]:
# Extract multiple groups at once using named groups
# Pattern: Name - Email - Phone
full_pattern = r'(?P<name>[\w\s]+) - (?P<email>[\w\.]+@[\w\.]+) - (?P<phone>01\d-\d{7})'

extracted = contacts['info'].str.extract(full_pattern)
extracted

## 4. Data Cleaning with `.str.replace()`

Clean and standardize data by replacing patterns.

In [ ]:
# Sample: Messy price data
prices = pd.DataFrame({
    'item': ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard'],
    'price_raw': ['RM 2,500.00', 'RM1,200', 'RM 899.99', 'RM800.00', 'RM 150']
})

prices

In [ ]:
# Clean prices: remove RM, spaces, and commas
prices['price_clean'] = prices['price_raw'].str.replace(r'[RM\s,]', '', regex=True)
prices['price_clean'] = prices['price_clean'].astype(float)

prices

In [ ]:
# Sample: Standardize phone numbers
phones = pd.Series([
    '+60 12-345 6789',
    '012 3456789',
    '60123456789',
    '012-345-6789',
    '+6012 345 6789'
])

# Remove all non-digits, then format consistently
phones_clean = phones.str.replace(r'\D', '', regex=True)  # Remove non-digits
phones_clean = phones_clean.str.replace(r'^60', '0', regex=True)  # 60 -> 0 prefix

print("Original:")
print(phones.tolist())
print("\nCleaned:")
print(phones_clean.tolist())

## 5. Splitting Text with `.str.split()`

Split strings into lists or expand into multiple columns.

In [ ]:
# Sample: Full names
names = pd.DataFrame({
    'full_name': [
        'John Michael Smith',
        'Jane Doe',
        'Robert James Wilson Jr',
        'Alice Lee'
    ]
})

names

In [ ]:
# Split into first and last name (expand=True creates columns)
# n=1 limits to 2 parts (first split only)
names[['first_name', 'rest']] = names['full_name'].str.split(r'\s+', n=1, expand=True)

names

In [ ]:
# Split addresses
addresses = pd.Series([
    '123 Main St, Kuala Lumpur, 50000',
    '45 Jalan Raja, Petaling Jaya, 46000',
    '789 Lorong Utara, Georgetown, 10200'
])

# Split by comma (with optional spaces)
address_parts = addresses.str.split(r',\s*', expand=True)
address_parts.columns = ['street', 'city', 'postcode']
address_parts

## 6. Real-World Data Cleaning Example

Let's work through a complete data cleaning scenario.

In [ ]:
# Messy customer data (common real-world issues)
messy_data = pd.DataFrame({
    'customer_record': [
        'ID:C001 | Name: John Smith | IC: 880512-08-5544 | Tel: 012-345 6789',
        'ID:C002 | Name: Jane Doe | IC: 920315145566 | Tel: +60 11-9876543',
        'ID:C003 | Name: Bob wilson | IC: 750821-10-1234 | Tel: 013 222 3333',
        'ID:C004 | Name: ALICE LEE | IC: 881025141122 | Tel: 019-4445556',
        'ID:C005 | Name: Charlie Tan | IC: 950607-14-7788 | Tel: 012 111-2222'
    ]
})

print("Original messy data:")
messy_data

In [ ]:
# Step 1: Extract customer ID
messy_data['customer_id'] = messy_data['customer_record'].str.extract(r'ID:(C\d+)')

# Step 2: Extract name and standardize to Title Case
messy_data['name'] = messy_data['customer_record'].str.extract(r'Name:\s*([^|]+)')
messy_data['name'] = messy_data['name'].str.strip().str.title()

# Step 3: Extract IC number and standardize format (XXXXXX-XX-XXXX)
messy_data['ic_raw'] = messy_data['customer_record'].str.extract(r'IC:\s*(\d{6}-?\d{2}-?\d{4})')
messy_data['ic_clean'] = messy_data['ic_raw'].str.replace(r'(\d{6})-?(\d{2})-?(\d{4})', 
                                                          r'\1-\2-\3', regex=True)

# Step 4: Extract phone and standardize format
messy_data['phone_raw'] = messy_data['customer_record'].str.extract(r'Tel:\s*([^|]+)')
messy_data['phone_clean'] = messy_data['phone_raw'].str.replace(r'\D', '', regex=True)
messy_data['phone_clean'] = messy_data['phone_clean'].str.replace(r'^60', '0', regex=True)

# Show results
messy_data[['customer_id', 'name', 'ic_clean', 'phone_clean']]

In [ ]:
# Step 5: Extract birth date from IC number
# Malaysian IC format: YYMMDD-PB-XXXX (PB = place of birth)

def extract_birthdate(ic):
    """Extract birth date from Malaysian IC number."""
    match = re.match(r'(\d{2})(\d{2})(\d{2})', ic.replace('-', ''))
    if match:
        yy, mm, dd = match.groups()
        # Determine century (00-30 = 2000s, 31-99 = 1900s)
        year = int(yy)
        century = '20' if year <= 30 else '19'
        return f"{century}{yy}-{mm}-{dd}"
    return None

messy_data['birth_date'] = messy_data['ic_clean'].apply(extract_birthdate)
messy_data[['name', 'ic_clean', 'birth_date']]

## 7. Validating Data with Regex

Use regex to validate data formats.

In [ ]:
# Sample: Email validation
emails = pd.DataFrame({
    'email': [
        'valid@email.com',
        'also.valid@company.org',
        'invalid-email',
        'missing@domain',
        '@nodomain.com',
        'good_one123@mail.co.uk'
    ]
})

# Basic email validation pattern
email_pattern = r'^[\w\.-]+@[\w\.-]+\.[a-zA-Z]{2,}$'

emails['is_valid'] = emails['email'].str.match(email_pattern)
emails

In [ ]:
# Validate Malaysian IC numbers
ic_numbers = pd.Series([
    '880512-08-5544',   # Valid
    '920315145566',     # Valid (no dashes)
    '880512085544',     # Valid (no dashes)
    '88-0512-08-5544',  # Invalid format
    '8805120855',       # Too short
    '880512-08-554A'    # Contains letter
])

# Pattern: 6 digits - 2 digits - 4 digits (dashes optional)
ic_pattern = r'^\d{6}-?\d{2}-?\d{4}$'

valid_mask = ic_numbers.str.match(ic_pattern)
pd.DataFrame({'IC': ic_numbers, 'Valid': valid_mask})

## 8. Finding All Matches with `.str.findall()`

Extract all occurrences of a pattern (not just the first).

In [ ]:
# Sample: Text with multiple tags/hashtags
posts = pd.DataFrame({
    'post': [
        'Great day at #KualaLumpur #Malaysia #travel',
        'Learning #Python and #DataScience today!',
        'No hashtags here.',
        '#Morning #Coffee #Coding #Productivity'
    ]
})

# Find all hashtags
posts['hashtags'] = posts['post'].str.findall(r'#\w+')
posts['hashtag_count'] = posts['hashtags'].str.len()

posts

In [ ]:
# Sample: Extract all numbers from text
text_data = pd.Series([
    'Order #12345: 3 items at RM99.50 each',
    'Invoice 2024-001: Total RM1,500.00',
    'Ref: ABC-789, Qty: 15, Price: 25.00'
])

# Find all numbers (including decimals)
numbers = text_data.str.findall(r'\d+\.?\d*')
numbers

## 9. Case Study: Cleaning Product Data

Let's clean a realistic product dataset.

In [ ]:
# Messy product data
products = pd.DataFrame({
    'product_info': [
        'SKU: PRD-001 | Apple iPhone 14 Pro (256GB) - Space Black | Price: RM5,499.00',
        'SKU:PRD-002|Samsung Galaxy S23 Ultra 512GB Phantom Black|Price:RM 6,299',
        'SKU: PRD-003 | Sony WH-1000XM5 Headphones | Price: RM 1,699.00',
        'SKU:PRD-004| Apple MacBook Pro 14" M3 Pro|Price: RM9999',
        'SKU: PRD-005 | Dell XPS 15 (2024) i7 32GB | Price: RM 8,500.00'
    ]
})

products

In [ ]:
# Extract all fields using named groups
pattern = r'SKU:\s?(?P<sku>PRD-\d+)\s?\|\s?(?P<name>[^|]+)\s?\|\s?Price:\s?RM\s?(?P<price>[\d,\.]+)'

extracted = products['product_info'].str.extract(pattern)
extracted

In [ ]:
# Clean and process the extracted data
extracted['name'] = extracted['name'].str.strip()
extracted['price'] = extracted['price'].str.replace(',', '').astype(float)

# Extract brand from name (first word)
extracted['brand'] = extracted['name'].str.extract(r'^(\w+)')

# Extract storage capacity if present
extracted['storage'] = extracted['name'].str.extract(r'(\d+GB|\d+TB)', flags=re.IGNORECASE)

extracted

## 10. Common Data Cleaning Patterns Reference

Quick reference for common cleaning tasks.

In [ ]:
# Common cleaning patterns
sample = pd.Series([
    '  Extra   Spaces  Here  ',
    'SHOUTY TEXT',
    'lowercase text',
    'MixedCase123Text',
    'Special@#$Characters!',
    '  123-456-7890  '
])

print("Original:")
print(sample.tolist())
print()

# 1. Remove extra whitespace
print("Remove extra spaces:")
print(sample.str.replace(r'\s+', ' ', regex=True).str.strip().tolist())
print()

# 2. Remove non-alphanumeric characters
print("Keep only alphanumeric:")
print(sample.str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.strip().tolist())
print()

# 3. Extract only digits
print("Extract digits only:")
print(sample.str.replace(r'\D', '', regex=True).tolist())
print()

# 4. Extract only letters
print("Extract letters only:")
print(sample.str.replace(r'[^a-zA-Z]', '', regex=True).tolist())

## Quick Reference: Pandas Regex Methods

| Task | Code |
|------|------|
| Filter rows containing pattern | `df[df['col'].str.contains(r'pattern', na=False)]` |
| Extract first match | `df['col'].str.extract(r'(pattern)')` |
| Extract with named groups | `df['col'].str.extract(r'(?P<name>pattern)')` |
| Find all matches | `df['col'].str.findall(r'pattern')` |
| Replace pattern | `df['col'].str.replace(r'pattern', 'repl', regex=True)` |
| Split by pattern | `df['col'].str.split(r'pattern', expand=True)` |
| Validate format | `df['col'].str.match(r'^pattern$')` |
| Case insensitive | `flags=re.IGNORECASE` or `case=False` |

---

## Summary

In this notebook, you learned:

1. **Filtering** - Use `.str.contains()` to filter DataFrames by pattern
2. **Extracting** - Use `.str.extract()` with capturing groups to pull structured data
3. **Replacing** - Use `.str.replace()` to clean and standardize data
4. **Splitting** - Use `.str.split()` to break text into columns
5. **Validating** - Use `.str.match()` to verify data formats
6. **Finding all** - Use `.str.findall()` for multiple matches

### Next Steps
Complete the exercises in `03_RegEx_Exercises.ipynb` to practice these skills!